<a href="https://colab.research.google.com/github/Michelle110803/Data-Visualisation/blob/main/Data_Visualisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install plotly ipywidgets

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from ipywidgets import interact, widgets
from IPython.display import clear_output, display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path = "/content/drive/MyDrive/Data Vis/owid-co2-data.csv"


In [5]:
df_raw = pd.read_csv(path)
df_raw.head()

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
#columns used for CO2 + Energy + GDP dashboard

cols = [
    "country", "iso_code", "year", "co2", "co2_per_capita", "gdp",
    "population", "primary_energy_consumption", "coal_co2",
    "oil_co2", "gas_co2"
]

df = df_raw[cols].copy()

#filter years >=1990 (better coverage + easier trend discussion)
df = df[df["year"] >= 1990].copy()

#keep only real countries
df = df[df["iso_code"].notna()].copy()
df = df[df["iso_code"].str.fullmatch(r"[A-Z]{3}")].copy()

#remove OWID aggregate regions
df = df[~df["iso_code"].astype(str).str.startswith("OWID_")].copy()

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7630 entries, 240 to 50410
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   country                     7630 non-null   object 
 1   iso_code                    7630 non-null   object 
 2   year                        7630 non-null   int64  
 3   co2                         7508 non-null   float64
 4   co2_per_capita              7442 non-null   float64
 5   gdp                         5410 non-null   float64
 6   population                  7560 non-null   float64
 7   primary_energy_consumption  6959 non-null   float64
 8   coal_co2                    4752 non-null   float64
 9   oil_co2                     7452 non-null   float64
 10  gas_co2                     4257 non-null   float64
dtypes: float64(8), int64(1), object(2)
memory usage: 715.3+ KB


In [7]:
#line chart: global CO2 trend
global_ts = (
    df.dropna(subset=["co2"])
    .groupby("year", as_index = False)["co2"]
    .sum()
)

fig = px.line(
    global_ts,
    x = "year",
    y = "co2",
    title = "Global CO2 emissions over time (1990 onwards )"
)

fig.update_layout(yaxis_title = "CO2 (million tonnes)")
fig.show()


In [8]:
#Chloropeth map : CO2 by country
map_df = df.dropna(subset=["iso_code", "co2"])

fig = px.choropleth(
    map_df,
    locations = "iso_code",
    color = "co2",
    hover_name = "country",
    animation_frame = "year",
    color_continuous_scale = "Reds",
    title = "CO2 emissions by country (1990 onwards)"
)

fig.update_layout(margin=dict(l = 0, r = 0, t = 60, b = 0))
fig.show()

In [9]:
#scatter: GDP vs CO2 (year slider)

scatter_df = df.dropna(subset=["gdp", "co2", "population"])

fig = px.scatter(
    scatter_df,
    x = "gdp",
    y = "co2",
    size = "population",
    hover_name = "country",
    animation_frame = "year",
    animation_group = "country",
    log_x = True,
    title = "GDP vs CO2 emissions (animated by year)",
    labels = {"gdp" : "GDP", "co2" : "CO2 (million tonnes)"}
)

fig.show()

In [10]:
import plotly.graph_objects as go

# stacked area: Fossil CO2 mix (coal / oil / gas) by country
mix_df = df.dropna(subset=["coal_co2", "oil_co2", "gas_co2"]).copy()

# Pick Top 30 countries by average total fossil CO2
mix_df["fossil_total"] = mix_df["coal_co2"] + mix_df["oil_co2"] + mix_df["gas_co2"]
top_countries = (
    mix_df.groupby("country")["fossil_total"]
    .mean()
    .sort_values(ascending=False)
    .head(30)
    .index
    .tolist()
)

mix_top = mix_df[mix_df["country"].isin(top_countries)].copy()
top_countries = sorted(top_countries)

fig = go.Figure()
trace_map = {}

for c in top_countries:
    d = mix_top[mix_top["country"] == c].sort_values("year")
    start = len(fig.data)

    fig.add_trace(go.Scatter(
        x=d["year"],
        y=d["coal_co2"],
        stackgroup="one",
        name="Coal CO₂",
        visible=False,
        mode="lines",
        line=dict(color="royalblue", width=2),
        fillcolor="rgba(65, 105, 225, 0.45)"
    ))

    fig.add_trace(go.Scatter(
        x=d["year"],
        y=d["oil_co2"],
        stackgroup="one",
        name="Oil CO₂",
        visible=False,
        mode="lines",
        line=dict(color="tomato", width=2),
        fillcolor="rgba(255, 99, 71, 0.45)"
    ))

    fig.add_trace(go.Scatter(
        x=d["year"],
        y=d["gas_co2"],
        stackgroup="one",
        name="Gas CO₂",
        visible=False,
        mode="lines",
        line=dict(color="seagreen", width=2),
        fillcolor="rgba(46, 139, 87, 0.45)"
    ))

    trace_map[c] = [start, start + 1, start + 2]

# default country
default = "Australia" if "Australia" in top_countries else (
    "Malaysia" if "Malaysia" in top_countries else top_countries[0]
)

for i in trace_map[default]:
    fig.data[i].visible = True

buttons = []
for c in top_countries:
    visible = [False] * len(fig.data)
    for i in trace_map[c]:
        visible[i] = True
    buttons.append(dict(
        label=c,
        method="update",
        args=[
            {"visible": visible},
            {"title": f"Fossil source CO₂ mix over time — {c} (Top 30 emitters)"}
        ]
    ))

fig.update_layout(
    title=f"Fossil source CO₂ mix over time — {default} (Top 30 emitters)",
    xaxis_title="Year",
    yaxis_title="CO₂ (million tonnes)",
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top"
    )],
    legend=dict(
        x=1.02,
        y=0.82,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)"
    ),
    margin=dict(l=60, r=260, t=80, b=60)
)

fig.show()

In [11]:
#bar chart

bar_df = df.dropna(subset=["country", "year", "co2"]).copy()

fig = px.bar(
    bar_df,
    x = "country",
    y = "co2",
    animation_frame = "year",
    title = "CO2 emissions by country over time",
    labels = {"co2" : "CO2 (million tonnes)", "country" : "Country"}
)

fig.update_layout(
    xaxis_title = "Country",
    yaxis_title = "CO2 (million tonnes)"
)

fig.show()

In [12]:
import numpy as np

z = np.polyfit(global_ts["year"], global_ts["co2"],1)
p = np.poly1d(z)

global_ts["trend"] = p(global_ts["year"])

fig = px.line(
    global_ts,
    x = "year",
    y = ["co2","trend"],
    title = "Global CO2 emissions with trend projection"
)

fig.update_layout(yaxis_title="CO2 (million tonnes)", legend_title = "Variable")
fig.show()
